# Disease Terms Review

Notebook version of `review_disease_terms.py`. BioPortal is the primary source; OLS and direct OBO Foundry scans are additive and merged by class IRI. The CSV includes definitions, comments, synonyms, parent information, BFO anchors where found, and provenance columns for source database, retrieval time, ontology version, source ontology id, and source URL.


## 1. Imports And Configuration

The default CSV output is `Scoping review/disease_terms_review.csv`. Check `ENV_FILE` if the BioPortal key lives somewhere else. For OLS tunnel setups, edit `OLS_BASE_URL_NOTEBOOK` in the run cell at the bottom.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import sys
import time
import urllib.parse
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
from typing import Any

import requests

# -- Config --------------------------------------------------------------------

ENV_FILE = Path(
    "/Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Forschung/Ontology/Wound ontology/.env"
)

BIOPORTAL_BASE_URL = "https://data.bioontology.org"
OLS_BASE_URL = "https://www.ebi.ac.uk/ols4"
OBO_REGISTRY_URL = "https://obofoundry.org/registry/ontologies.yml"
SCOPING_REVIEW_DIR = Path("Scoping review")
DEFAULT_OUT = SCOPING_REVIEW_DIR / "disease_terms_review.csv"

CACHE_DIR = Path("/tmp/disease_terms_review_cache")

DEFAULT_TERMS = [
    "disease",
    "disorder",
    "injury",
    "illness",
    "condition",
    "syndrome",
    "pathological process",
    "pathological disposition",
    "dysfunction",
    "predisposition",
]

BFO_PREFIX = "http://purl.obolibrary.org/obo/BFO_"
DELAY = 0.12

DEFINITION_KEYS = {
    "definition",
    "description",
    "iao_0000115",
    "http://purl.obolibrary.org/obo/iao_0000115",
}
COMMENT_KEYS = {
    "comment",
    "rdfs:comment",
    "http://www.w3.org/2000/01/rdf-schema#comment",
}

## 2. Generic Helpers

In [2]:
# -- Generic helpers -----------------------------------------------------------

def now_iso() -> str:
    return datetime.now().astimezone().isoformat(timespec="seconds")


def clean_list(values: Any) -> list[str]:
    if not values:
        return []
    if isinstance(values, str):
        values = [values]
    return [str(v).strip() for v in values if str(v).strip()]


def first_clean(values: Any) -> str:
    values = clean_list(values)
    return values[0] if values else ""


def join_values(values: Any) -> str:
    seen: set[str] = set()
    out: list[str] = []
    for value in clean_list(values):
        if value not in seen:
            out.append(value)
            seen.add(value)
    return "; ".join(out)


def append_unique(existing: str, value: str) -> str:
    parts = clean_list(existing.split(";")) if existing else []
    if value and value not in parts:
        parts.append(value)
    return "; ".join(parts)


def cache_path(namespace: str, url: str, params: dict[str, Any] | None = None) -> Path:
    params = params or {}
    serialised = json.dumps([url, sorted(params.items())], sort_keys=True, default=str)
    digest = hashlib.sha256(serialised.encode("utf-8")).hexdigest()
    path = CACHE_DIR / namespace
    path.mkdir(parents=True, exist_ok=True)
    return path / f"{digest}.json"


def get_json(
    url: str,
    params: dict[str, Any] | None = None,
    *,
    namespace: str = "generic",
    retries: int = 3,
    headers: dict[str, str] | None = None,
    cache: bool = True,
) -> Any | None:
    path = cache_path(namespace, url, params)
    if cache and path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    for attempt in range(retries):
        try:
            time.sleep(DELAY)
            response = requests.get(url, params=params or {}, headers=headers, timeout=60)
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 10))
                print(f"  [rate limit] waiting {wait}s", file=sys.stderr)
                time.sleep(wait)
                continue
            response.raise_for_status()
            data = response.json()
            if cache:
                path.write_text(json.dumps(data), encoding="utf-8")
            return data
        except Exception as exc:
            print(f"  [warn] GET {url} attempt {attempt + 1} failed: {exc}", file=sys.stderr)
            time.sleep(2 ** attempt)
    return None


def get_text(
    url: str,
    *,
    namespace: str,
    retries: int = 3,
    cache: bool = True,
) -> str | None:
    path = cache_path(namespace, url)
    text_path = path.with_suffix(".txt")
    if cache and text_path.exists():
        return text_path.read_text(encoding="utf-8", errors="replace")

    for attempt in range(retries):
        try:
            time.sleep(DELAY)
            response = requests.get(url, timeout=120)
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 10))
                print(f"  [rate limit] waiting {wait}s", file=sys.stderr)
                time.sleep(wait)
                continue
            response.raise_for_status()
            text = response.text
            if cache:
                text_path.write_text(text, encoding="utf-8")
            return text
        except Exception as exc:
            print(f"  [warn] GET {url} attempt {attempt + 1} failed: {exc}", file=sys.stderr)
            time.sleep(2 ** attempt)
    return None


def bfo_anchor(parents: list[dict[str, str]]) -> str:
    for parent in parents:
        iri = parent.get("@id", "") or ""
        if iri.startswith(BFO_PREFIX):
            label = (parent.get("prefLabel") or "").strip()
            return f"{label} ({iri.split('BFO_')[1]})" if label else iri
    return ""


def obo_curie_to_iri(curie: str) -> str:
    return f"http://purl.obolibrary.org/obo/{curie.replace(':', '_')}"


def annotation_key_matches(key: str, wanted_keys: set[str]) -> bool:
    lowered = key.strip().lower()
    local = lowered.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
    return lowered in wanted_keys or local in wanted_keys


def extract_annotation_values(data: Any, wanted_keys: set[str]) -> list[str]:
    values: list[str] = []
    if not isinstance(data, dict):
        return values

    for key, value in data.items():
        if annotation_key_matches(str(key), wanted_keys):
            values.extend(clean_list(value))

    for container_key in ["properties", "annotation", "annotations"]:
        container = data.get(container_key)
        if isinstance(container, dict):
            for key, value in container.items():
                if annotation_key_matches(str(key), wanted_keys):
                    values.extend(clean_list(value))
        elif isinstance(container, list):
            for item in container:
                values.extend(extract_annotation_values(item, wanted_keys))

    return values


def make_row(
    *,
    term: str,
    source: str,
    ontology: str,
    ontology_version: str = "",
    preferred_label: str = "",
    class_iri: str = "",
    definition: str = "",
    comments: str = "",
    synonyms: str = "",
    parent_labels: str = "",
    parent_iris: str = "",
    bfo_anchor_value: str = "",
    mapping_count: str | int = "",
    source_url: str = "",
    retrieved_at: str | None = None,
) -> dict[str, Any]:
    retrieved_at = retrieved_at or now_iso()
    row = {
        "term": term,
        "ontology": ontology,
        "preferred_label": preferred_label,
        "class_iri": class_iri,
        "definition": definition,
        "comments": comments,
        "synonyms": synonyms,
        "parent_labels": parent_labels,
        "parent_iris": parent_iris,
        "bfo_anchor": bfo_anchor_value,
        "mapping_count": mapping_count,
        "source_databases": source,
        "retrieved_at_by_database": f"{source}: {retrieved_at}",
        "ontology_versions": f"{source}: {ontology_version}" if ontology_version else "",
        "source_ontology_ids": f"{source}: {ontology}" if ontology else "",
        "source_urls": f"{source}: {source_url}" if source_url else "",
        "source_records": json.dumps(
            [
                {
                    "source": source,
                    "retrieved_at": retrieved_at,
                    "ontology": ontology,
                    "ontology_version": ontology_version,
                    "source_url": source_url,
                }
            ],
            ensure_ascii=False,
        ),
    }
    return row


def dedupe_key(row: dict[str, Any]) -> tuple[str, str]:
    iri = str(row.get("class_iri", "")).strip()
    if iri:
        return ("iri", iri)
    label_key = "|".join(
        [
            str(row.get("ontology", "")).lower(),
            str(row.get("preferred_label", "")).lower(),
            str(row.get("term", "")).lower(),
        ]
    )
    return ("fallback", label_key)


def merge_row(existing: dict[str, Any], incoming: dict[str, Any]) -> None:
    fill_if_blank = [
        "ontology",
        "preferred_label",
        "class_iri",
        "synonyms",
        "parent_labels",
        "parent_iris",
        "bfo_anchor",
        "mapping_count",
    ]
    for field in fill_if_blank:
        if not existing.get(field) and incoming.get(field) not in (None, ""):
            existing[field] = incoming[field]

    for field in ["definition", "comments"]:
        for part in clean_list(str(incoming.get(field, "")).split(";")):
            existing[field] = append_unique(str(existing.get(field, "")), part)

    for field in [
        "source_databases",
        "retrieved_at_by_database",
        "ontology_versions",
        "source_ontology_ids",
        "source_urls",
    ]:
        for part in clean_list(str(incoming.get(field, "")).split(";")):
            existing[field] = append_unique(str(existing.get(field, "")), part)

    try:
        existing_records = json.loads(existing.get("source_records", "[]") or "[]")
        incoming_records = json.loads(incoming.get("source_records", "[]") or "[]")
        seen = {(r.get("source"), r.get("ontology"), r.get("retrieved_at")) for r in existing_records}
        for record in incoming_records:
            key = (record.get("source"), record.get("ontology"), record.get("retrieved_at"))
            if key not in seen:
                existing_records.append(record)
                seen.add(key)
        existing["source_records"] = json.dumps(existing_records, ensure_ascii=False)
    except json.JSONDecodeError:
        existing["source_records"] = incoming.get("source_records", existing.get("source_records", ""))


def merge_rows(rows_by_key: dict[tuple[str, str], dict[str, Any]], rows: list[dict[str, Any]]) -> None:
    for row in rows:
        key = dedupe_key(row)
        if key in rows_by_key:
            merge_row(rows_by_key[key], row)
        else:
            rows_by_key[key] = row

## 3. BioPortal Source

BioPortal runs first when selected. By default, raw BioPortal results are filtered to exact preferred-label matches. Definitions and comments are read from BioPortal fields and class properties when available.

In [3]:
# -- BioPortal -----------------------------------------------------------------

def load_bioportal_api_key() -> str:
    key = os.environ.get("BIOPORTAL_KEY", "")
    if not key and ENV_FILE.exists():
        for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
            if line.startswith("BIOPORTAL_KEY="):
                key = line.split("=", 1)[1].strip()
    if not key:
        sys.exit("ERROR: BIOPORTAL_KEY not found")
    return key


def bioportal_get(url: str, params: dict[str, Any], apikey: str) -> Any | None:
    request_params = {**params, "apikey": apikey}
    return get_json(url, request_params, namespace="bioportal")


def fetch_bioportal_ontology_ids(apikey: str) -> set[str]:
    """Return lowercased acronyms of every ontology indexed in BioPortal (cached)."""
    data = get_json(
        f"{BIOPORTAL_BASE_URL}/ontologies",
        {"apikey": apikey, "display_links": "false", "display_context": "false"},
        namespace="bioportal",
    )
    if not isinstance(data, list):
        return set()
    ids = {str(o.get("acronym", "")).lower() for o in data if o.get("acronym")}
    print(f"   BioPortal indexes {len(ids)} ontologies — OLS/OBO scan will skip these", file=sys.stderr)
    return ids


def search_bioportal_exact(term: str, apikey: str) -> list[dict[str, Any]]:
    hits: list[dict[str, Any]] = []
    page = 1
    while True:
        data = bioportal_get(
            f"{BIOPORTAL_BASE_URL}/search",
            {
                "q": term,
                "require_exact_match": "true",
                "include": "prefLabel,definition,synonym,notation,properties",
                "pagesize": 100,
                "page": page,
            },
            apikey,
        )
        if not data:
            break
        hits.extend(data.get("collection", []))
        if not data.get("links", {}).get("nextPage"):
            break
        page += 1
    return hits


def fetch_bioportal_parents(ontology: str, class_iri: str, apikey: str) -> list[dict[str, Any]]:
    encoded = urllib.parse.quote(class_iri, safe="")
    data = bioportal_get(
        f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{encoded}/parents",
        {"include": "prefLabel"},
        apikey,
    )
    return data if isinstance(data, list) else []


def fetch_bioportal_mapping_count(ontology: str, class_iri: str, apikey: str) -> int:
    encoded = urllib.parse.quote(class_iri, safe="")
    data = bioportal_get(
        f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{encoded}/mappings",
        {},
        apikey,
    )
    if isinstance(data, list):
        return len(data)
    if isinstance(data, dict):
        return int(data.get("totalCount", len(data.get("collection", []))))
    return 0


def fetch_bioportal_ontology_version(ontology: str, apikey: str) -> str:
    data = bioportal_get(f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/latest_submission", {}, apikey)
    if not isinstance(data, dict):
        return ""
    parts = []
    for key in ["version", "released", "creationDate", "submissionId"]:
        value = data.get(key)
        if value:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


def bioportal_rows(
    terms: list[str],
    apikey: str,
    *,
    fetch_parents: bool,
    fetch_mappings: bool,
    exact_label_only: bool,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    version_cache: dict[str, str] = {}

    for term in terms:
        print(f"\n-- BioPortal: searching '{term}' --", file=sys.stderr)
        hits = search_bioportal_exact(term, apikey)
        raw_count = len(hits)
        if exact_label_only:
            term_lower = term.casefold()
            hits = [hit for hit in hits if str(hit.get("prefLabel", "")).casefold() == term_lower]
        if exact_label_only:
            print(f"   {len(hits)} exact-label hits ({raw_count} raw BioPortal hits)", file=sys.stderr)
        else:
            print(f"   {len(hits)} hits", file=sys.stderr)

        for hit in hits:
            retrieved_at = now_iso()
            iri = hit.get("@id", "")
            ontology = hit.get("links", {}).get("ontology", "").rstrip("/").split("/")[-1]
            label = hit.get("prefLabel", "").strip()
            definition = join_values(clean_list(hit.get("definition")) + extract_annotation_values(hit, DEFINITION_KEYS))
            comments = join_values(extract_annotation_values(hit, COMMENT_KEYS))
            synonyms = join_values(hit.get("synonym"))

            parent_labels = parent_iris = anchor = ""
            mapping_count: str | int = ""

            if ontology and ontology not in version_cache:
                version_cache[ontology] = fetch_bioportal_ontology_version(ontology, apikey)

            if fetch_parents and ontology and iri:
                parents = fetch_bioportal_parents(ontology, iri, apikey)
                parent_labels = join_values(p.get("prefLabel") for p in parents)
                parent_iris = join_values(p.get("@id") for p in parents)
                anchor = bfo_anchor([{"@id": p.get("@id", ""), "prefLabel": p.get("prefLabel", "")} for p in parents])
                if fetch_mappings:
                    mapping_count = fetch_bioportal_mapping_count(ontology, iri, apikey)

            rows.append(
                make_row(
                    term=term,
                    source="BioPortal",
                    ontology=ontology,
                    ontology_version=version_cache.get(ontology, ""),
                    preferred_label=label,
                    class_iri=iri,
                    definition=definition,
                    comments=comments,
                    synonyms=synonyms,
                    parent_labels=parent_labels,
                    parent_iris=parent_iris,
                    bfo_anchor_value=anchor,
                    mapping_count=mapping_count,
                    source_url=f"{BIOPORTAL_BASE_URL}/ontologies/{ontology}/classes/{urllib.parse.quote(iri, safe='')}" if ontology and iri else "",
                    retrieved_at=retrieved_at,
                )
            )
            note = f" [BFO: {anchor}]" if anchor else ""
            print(f"   {ontology:15s} {label}{note}", file=sys.stderr)
    return rows

## 4. OLS Source

OLS uses `/ols4/api/search` with label-scoped exact class search and then fetches term details to capture definitions/comments from annotations where available.

In [4]:
# -- OLS -----------------------------------------------------------------------

def normalise_ols_base_url(base_url: str) -> str:
    base_url = base_url.rstrip("/")
    return base_url if base_url.endswith("/ols4") else f"{base_url}/ols4"


def ols_get(base_url: str, path: str, params: dict[str, Any] | None = None) -> Any | None:
    return get_json(f"{base_url.rstrip('/')}{path}", params or {}, namespace="ols")


def ols_iri_variants(iri: str) -> list[str]:
    once = urllib.parse.quote(iri, safe="")
    slash_double_encoded = once.replace("%2F", "%252F")
    twice = urllib.parse.quote(once, safe="")
    variants = []
    for value in [slash_double_encoded, once, twice]:
        if value not in variants:
            variants.append(value)
    return variants


def search_ols_exact(term: str, base_url: str) -> list[dict[str, Any]]:
    docs: list[dict[str, Any]] = []
    rows = 100
    start = 0

    while True:
        data = ols_get(
            base_url,
            "/api/search",
            {
                "q": term,
                "exact": "true",
                "type": "class",
                "queryFields": "label",
                "fieldList": "iri,label,ontology_name,ontology_prefix,description,comment,annotation,synonym,exact_synonyms,obo_id,type",
                "rows": rows,
                "start": start,
            },
        )
        response = data.get("response", {}) if isinstance(data, dict) else {}
        batch = response.get("docs", [])
        docs.extend(batch)
        num_found = int(response.get("numFound", len(docs)))
        start += rows
        if start >= num_found or not batch:
            break

    term_lower = term.casefold()
    return [doc for doc in docs if str(doc.get("label", "")).casefold() == term_lower]


def fetch_ols_ontology_version(ontology: str, base_url: str) -> str:
    data = ols_get(base_url, f"/api/ontologies/{ontology}")
    if not isinstance(data, dict):
        return ""
    config = data.get("config") or {}
    parts = []
    for key in ["version", "versionIri", "title", "namespace"]:
        value = config.get(key) or data.get(key)
        if value:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


def extract_ols_terms(data: Any) -> list[dict[str, Any]]:
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        return []
    embedded = data.get("_embedded", {})
    for key in ["terms", "classes"]:
        if isinstance(embedded.get(key), list):
            return embedded[key]
    if isinstance(data.get("terms"), list):
        return data["terms"]
    if data.get("iri") or data.get("label"):
        return [data]
    return []


def fetch_ols_parents(ontology: str, iri: str, base_url: str) -> list[dict[str, str]]:
    for encoded in ols_iri_variants(iri):
        data = ols_get(base_url, f"/api/ontologies/{ontology}/terms/{encoded}/parents")
        terms = extract_ols_terms(data)
        if terms:
            return [
                {
                    "@id": parent.get("iri", ""),
                    "prefLabel": parent.get("label", ""),
                }
                for parent in terms
            ]
    return []


def fetch_ols_term_details(ontology: str, iri: str, base_url: str) -> dict[str, Any]:
    for encoded in ols_iri_variants(iri):
        data = ols_get(base_url, f"/api/ontologies/{ontology}/terms/{encoded}")
        if isinstance(data, dict) and data:
            return data
    return {}


def ols_rows(
    terms: list[str],
    *,
    base_url: str,
    fetch_parents: bool,
    skip_ids: set[str] | None = None,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    version_cache: dict[str, str] = {}

    for term in terms:
        print(f"\n-- OLS: searching '{term}' --", file=sys.stderr)
        hits = search_ols_exact(term, base_url)
        if skip_ids:
            before = len(hits)
            hits = [h for h in hits if (h.get("ontology_name") or h.get("ontology_prefix") or "").lower() not in skip_ids]
            skipped = before - len(hits)
            print(f"   {len(hits)} hits ({skipped} skipped — already in BioPortal)", file=sys.stderr)
        else:
            print(f"   {len(hits)} hits", file=sys.stderr)

        for hit in hits:
            retrieved_at = now_iso()
            iri = hit.get("iri", "")
            ontology = (hit.get("ontology_name") or hit.get("ontology_prefix") or "").lower()
            label = str(hit.get("label", "")).strip()
            detail = fetch_ols_term_details(ontology, iri, base_url) if ontology and iri else {}
            definition = join_values(
                clean_list(hit.get("description"))
                + clean_list(detail.get("description"))
                + extract_annotation_values(hit, DEFINITION_KEYS)
                + extract_annotation_values(detail, DEFINITION_KEYS)
            )
            comments = join_values(
                clean_list(hit.get("comment"))
                + clean_list(detail.get("comment"))
                + extract_annotation_values(hit, COMMENT_KEYS)
                + extract_annotation_values(detail, COMMENT_KEYS)
            )
            synonyms = join_values(clean_list(hit.get("synonym")) + clean_list(hit.get("exact_synonyms")))

            if ontology and ontology not in version_cache:
                version_cache[ontology] = fetch_ols_ontology_version(ontology, base_url)

            parent_labels = parent_iris = anchor = ""
            if fetch_parents and ontology and iri:
                parents = fetch_ols_parents(ontology, iri, base_url)
                parent_labels = join_values(p.get("prefLabel") for p in parents)
                parent_iris = join_values(p.get("@id") for p in parents)
                anchor = bfo_anchor(parents)

            rows.append(
                make_row(
                    term=term,
                    source="OLS",
                    ontology=ontology,
                    ontology_version=version_cache.get(ontology, ""),
                    preferred_label=label,
                    class_iri=iri,
                    definition=definition,
                    comments=comments,
                    synonyms=synonyms,
                    parent_labels=parent_labels,
                    parent_iris=parent_iris,
                    bfo_anchor_value=anchor,
                    source_url=f"{base_url}/api/ontologies/{ontology}/terms/{ols_iri_variants(iri)[0]}" if ontology and iri else f"{base_url}/api/search",
                    retrieved_at=retrieved_at,
                )
            )
            print(f"   {ontology:15s} {label}", file=sys.stderr)
    return rows

## 5. OBO Foundry Direct Source

This scans the OBO Foundry registry directly, preferring `.obo` products and falling back to OWL/RDF XML labels when needed. It extracts definitions and comments where available.

In [5]:
# -- OBO Foundry direct scan ---------------------------------------------------

def find_ontology_product_urls(record: dict[str, Any]) -> list[str]:
    urls: list[str] = []

    def add_url(url: str) -> None:
        if url and url not in urls:
            urls.append(url)

    for product in record.get("products", []) or []:
        url = product.get("ontology_purl") or product.get("url") or product.get("id") or ""
        if str(url).endswith(".obo"):
            add_url(str(url))
    for product in record.get("products", []) or []:
        url = product.get("ontology_purl") or product.get("url") or product.get("id") or ""
        if str(url).endswith(".owl"):
            add_url(str(url))
    url = record.get("ontology_purl") or ""
    if str(url).endswith((".obo", ".owl")):
        add_url(str(url))
    return urls


def fetch_obo_registry() -> list[dict[str, Any]]:
    text = get_text(OBO_REGISTRY_URL, namespace="obo-registry")
    if not text:
        return []
    return parse_obo_registry_yaml(text)


def parse_simple_yaml_value(value: str) -> str:
    value = value.strip()
    if value in {"", "|", ">"}:
        return ""
    return value.strip("\"'")


def parse_obo_registry_yaml(text: str) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    chunks = re.split(r"\n(?=- activity_status:)", text)
    for chunk in chunks:
        chunk = chunk.strip("\n")
        if not chunk.startswith("- activity_status:"):
            continue

        record: dict[str, Any] = {"products": []}
        lines = chunk.splitlines()
        for line in lines:
            if line.startswith("- activity_status:"):
                record["activity_status"] = parse_simple_yaml_value(line.split(":", 1)[1])
            elif match := re.match(r"^  (id|ontology_purl|title|preferredPrefix):\s*(.*)$", line):
                record[match.group(1)] = parse_simple_yaml_value(match.group(2))

        products_block = re.search(r"(?ms)^  products:\n(.*?)(?=^  [A-Za-z0-9_-]+:|\Z)", chunk)
        if products_block:
            current_product: dict[str, str] | None = None
            for line in products_block.group(1).splitlines():
                if line.startswith("  - "):
                    if current_product:
                        record["products"].append(current_product)
                    current_product = {}
                    rest = line[4:]
                    if ":" in rest:
                        key, value = rest.split(":", 1)
                        current_product[key.strip()] = parse_simple_yaml_value(value)
                elif current_product is not None and (match := re.match(r"^    (id|ontology_purl|title):\s*(.*)$", line)):
                    current_product[match.group(1)] = parse_simple_yaml_value(match.group(2))
            if current_product:
                record["products"].append(current_product)

        records.append(record)
    return records


def parse_obo_header(text: str) -> dict[str, str]:
    header: dict[str, str] = {}
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line == "[Term]":
            break
        if ":" in line:
            key, value = line.split(":", 1)
            header[key.strip()] = value.strip()
    return header


def parse_obo_stanza(stanza: dict[str, list[str]], terms_lower: set[str]) -> dict[str, Any] | None:
    names = clean_list(stanza.get("name"))
    if not names:
        return None
    name = names[0]
    if name.casefold() not in terms_lower:
        return None
    if first_clean(stanza.get("is_obsolete")).lower() == "true":
        return None

    curie = first_clean(stanza.get("id"))
    definition = ""
    if stanza.get("def"):
        match = re.search(r'"(.*?)"', stanza["def"][0])
        definition = match.group(1) if match else stanza["def"][0]
    comments = join_values(stanza.get("comment"))

    synonyms = []
    for synonym_line in stanza.get("synonym", []):
        match = re.search(r'"(.*?)"', synonym_line)
        if match:
            synonyms.append(match.group(1))

    parents: list[dict[str, str]] = []
    for parent_line in stanza.get("is_a", []):
        parent_curie = parent_line.split()[0]
        parent_label = parent_line.split("!", 1)[1].strip() if "!" in parent_line else ""
        parents.append({"@id": obo_curie_to_iri(parent_curie), "prefLabel": parent_label})

    return {
        "term": name,
        "preferred_label": name,
        "class_iri": obo_curie_to_iri(curie) if curie else "",
        "definition": definition,
        "comments": comments,
        "synonyms": join_values(synonyms),
        "parent_labels": join_values(p.get("prefLabel") for p in parents),
        "parent_iris": join_values(p.get("@id") for p in parents),
        "bfo_anchor": bfo_anchor(parents),
    }


def scan_obo_text(text: str, terms: list[str]) -> tuple[str, list[dict[str, Any]]]:
    header = parse_obo_header(text)
    version = header.get("data-version") or header.get("date") or header.get("format-version") or ""
    terms_lower = {term.casefold() for term in terms}

    rows: list[dict[str, Any]] = []
    current_kind = ""
    current: dict[str, list[str]] = {}

    def flush_current() -> None:
        if current_kind != "Term" or not current:
            return
        parsed = parse_obo_stanza(current, terms_lower)
        if parsed:
            rows.append(parsed)

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("!"):
            continue
        if line.startswith("[") and line.endswith("]"):
            flush_current()
            current_kind = line.strip("[]")
            current = {}
            continue
        if current_kind and ":" in line:
            key, value = line.split(":", 1)
            current.setdefault(key.strip(), []).append(value.strip())
    flush_current()
    return version, rows


def local_name(tag: str) -> str:
    return tag.rsplit("}", 1)[-1] if "}" in tag else tag


def child_texts(element: ET.Element, names: set[str]) -> list[str]:
    values = []
    for child in element:
        if local_name(child.tag) in names and child.text and child.text.strip():
            values.append(child.text.strip())
    return values


def scan_owl_xml_text(text: str, terms: list[str]) -> tuple[str, list[dict[str, Any]]]:
    try:
        root = ET.fromstring(text.encode("utf-8"))
    except ET.ParseError as exc:
        print(f"  [warn] OWL/XML parse failed: {exc}", file=sys.stderr)
        return "", []

    rdf_about = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about"
    rdf_resource = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource"
    owl_version_iri = "{http://www.w3.org/2002/07/owl#}versionIRI"
    owl_version_info = "{http://www.w3.org/2002/07/owl#}versionInfo"
    terms_lower = {term.casefold() for term in terms}

    version = ""
    for ontology in root.iter("{http://www.w3.org/2002/07/owl#}Ontology"):
        for child in ontology:
            if child.tag == owl_version_iri and child.attrib.get(rdf_resource):
                version = child.attrib[rdf_resource]
                break
            if child.tag == owl_version_info and child.text:
                version = child.text.strip()
                break
        if version:
            break

    rows: list[dict[str, Any]] = []
    class_elements = list(root.iter("{http://www.w3.org/2002/07/owl#}Class"))
    for element in class_elements:
        iri = element.attrib.get(rdf_about, "")
        if not iri:
            continue
        label = first_clean(child_texts(element, {"label"}))
        if not label or label.casefold() not in terms_lower:
            continue

        definitions = child_texts(element, {"IAO_0000115", "definition"})
        comments = child_texts(element, {"comment"})
        synonyms = []
        for child in element:
            name = local_name(child.tag)
            if "Synonym" in name and child.text and child.text.strip():
                synonyms.append(child.text.strip())

        parent_iris = []
        for child in element:
            if local_name(child.tag) == "subClassOf" and child.attrib.get(rdf_resource):
                parent_iris.append(child.attrib[rdf_resource])
        parents = [{"@id": parent_iri, "prefLabel": ""} for parent_iri in parent_iris]

        rows.append(
            {
                "term": label,
                "preferred_label": label,
                "class_iri": iri,
                "definition": first_clean(definitions),
                "comments": join_values(comments),
                "synonyms": join_values(synonyms),
                "parent_labels": "",
                "parent_iris": join_values(parent_iris),
                "bfo_anchor": bfo_anchor(parents),
            }
        )
    return version, rows


def scan_ontology_text(text: str, terms: list[str]) -> tuple[str, list[dict[str, Any]]]:
    stripped = text.lstrip()
    if stripped.startswith("format-version:") or "\n[Term]" in stripped[:5000]:
        return scan_obo_text(text, terms)
    if stripped.startswith("<?xml") or stripped.startswith("<rdf:RDF") or stripped.startswith("<Ontology"):
        return scan_owl_xml_text(text, terms)
    print("  [warn] ontology format not supported for direct OBO Foundry scan", file=sys.stderr)
    return "", []


def obo_rows(
    terms: list[str],
    *,
    max_ontologies: int | None,
    skip_ids: set[str] | None = None,
) -> list[dict[str, Any]]:
    print("\n-- OBO Foundry: loading registry --", file=sys.stderr)
    registry = fetch_obo_registry()
    active_records = [
        record for record in registry
        if str(record.get("activity_status", "active")).lower() not in {"inactive", "obsolete"}
    ]
    if skip_ids:
        before = len(active_records)
        active_records = [r for r in active_records if str(r.get("id", "")).lower() not in skip_ids]
        print(f"   skipped {before - len(active_records)} ontologies already indexed in BioPortal", file=sys.stderr)
    if max_ontologies is not None:
        active_records = active_records[:max_ontologies]
    print(f"   scanning {len(active_records)} OBO ontologies not in BioPortal", file=sys.stderr)

    rows: list[dict[str, Any]] = []
    for record in active_records:
        ontology = str(record.get("id", "")).lower()
        urls = find_ontology_product_urls(record)
        if not ontology or not urls:
            continue
        text = None
        url = ""
        for candidate_url in urls:
            print(f"   OBO {ontology}: {candidate_url}", file=sys.stderr)
            text = get_text(candidate_url, namespace="obo-files")
            if text:
                url = candidate_url
                break
        if not text:
            continue
        version, parsed_rows = scan_ontology_text(text, terms)
        retrieved_at = now_iso()
        for parsed in parsed_rows:
            rows.append(
                make_row(
                    term=parsed["term"],
                    source="OBO Foundry Direct",
                    ontology=ontology,
                    ontology_version=version,
                    preferred_label=parsed["preferred_label"],
                    class_iri=parsed["class_iri"],
                    definition=parsed["definition"],
                    comments=parsed["comments"],
                    synonyms=parsed["synonyms"],
                    parent_labels=parsed["parent_labels"],
                    parent_iris=parsed["parent_iris"],
                    bfo_anchor_value=parsed["bfo_anchor"],
                    source_url=url,
                    retrieved_at=retrieved_at,
                )
            )
    return rows

## 6. Merge And Output Helpers

In [6]:
# -- Main ----------------------------------------------------------------------

FIELDNAMES = [
    "term",
    "ontology",
    "preferred_label",
    "class_iri",
    "definition",
    "comments",
    "synonyms",
    "parent_labels",
    "parent_iris",
    "bfo_anchor",
    "mapping_count",
    "source_databases",
    "retrieved_at_by_database",
    "ontology_versions",
    "source_ontology_ids",
    "source_urls",
    "source_records",
]


def load_existing_rows(path: Path) -> dict[tuple[str, str], dict[str, Any]]:
    rows_by_key: dict[tuple[str, str], dict[str, Any]] = {}
    if not path.exists():
        return rows_by_key
    with path.open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            for field in FIELDNAMES:
                row.setdefault(field, "")
            rows_by_key[dedupe_key(row)] = row
    return rows_by_key


def parse_sources(raw_sources: str) -> list[str]:
    allowed = {"bioportal", "ols", "obo"}
    sources = [source.strip().lower() for source in raw_sources.split(",") if source.strip()]
    unknown = [source for source in sources if source not in allowed]
    if unknown:
        sys.exit(f"ERROR: unknown source(s): {', '.join(unknown)}")
    if "bioportal" not in sources:
        print("[warn] BioPortal is not selected; add-on sources will run without the primary pass.", file=sys.stderr)
    return sources

## 7. Run Three-Database Review

Edit the parameters in this cell before running. The default output goes into the `Scoping review/` folder.

In [7]:
# Run parameters - edit these before running the cell.
OUT_PATH = DEFAULT_OUT
SOURCES = "bioportal,ols,obo"  # options: bioportal, ols, obo
TERMS = DEFAULT_TERMS
OLS_BASE_URL_NOTEBOOK = os.environ.get("OLS_BASE_URL", OLS_BASE_URL)
OBO_MAX_ONTOLOGIES = None  # set to an integer for a small test run
FETCH_PARENTS = True
FETCH_MAPPINGS = True
INCLUDE_NONLABEL_MATCHES = False
RESUME = False


def write_rows(out_path: Path, rows_by_key: dict[tuple[str, str], dict[str, Any]]) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDNAMES, extrasaction="ignore")
        writer.writeheader()
        for row in sorted(
            rows_by_key.values(),
            key=lambda r: (r.get("term", ""), r.get("ontology", ""), r.get("class_iri", "")),
        ):
            writer.writerow(row)


def run_review(
    *,
    out_path: Path | str = OUT_PATH,
    sources: str | list[str] = SOURCES,
    terms: list[str] = TERMS,
    ols_base_url: str = OLS_BASE_URL_NOTEBOOK,
    obo_max_ontologies: int | None = OBO_MAX_ONTOLOGIES,
    fetch_parents: bool = FETCH_PARENTS,
    fetch_mappings: bool = FETCH_MAPPINGS,
    include_nonlabel_matches: bool = INCLUDE_NONLABEL_MATCHES,
    resume: bool = RESUME,
) -> list[dict[str, Any]]:
    source_text = ",".join(sources) if isinstance(sources, list) else sources
    selected_sources = parse_sources(source_text)
    selected_terms = terms or DEFAULT_TERMS
    out_path = Path(out_path)
    rows_by_key = load_existing_rows(out_path) if resume else {}

    if rows_by_key:
        print(f"[resume] loaded {len(rows_by_key)} existing merged rows", file=sys.stderr)

    # Fetch BioPortal ontology IDs once so OLS and OBO can skip redundant downloads.
    bp_ids: set[str] = set()
    if "bioportal" in selected_sources and ("ols" in selected_sources or "obo" in selected_sources):
        apikey = load_bioportal_api_key()
        bp_ids = fetch_bioportal_ontology_ids(apikey)

    for source in selected_sources:
        if source == "bioportal":
            apikey = load_bioportal_api_key()
            merge_rows(
                rows_by_key,
                bioportal_rows(
                    selected_terms,
                    apikey,
                    fetch_parents=fetch_parents,
                    fetch_mappings=fetch_mappings,
                    exact_label_only=not include_nonlabel_matches,
                ),
            )
        elif source == "ols":
            normalised_ols_base_url = normalise_ols_base_url(ols_base_url)
            merge_rows(
                rows_by_key,
                ols_rows(selected_terms, base_url=normalised_ols_base_url, fetch_parents=fetch_parents, skip_ids=bp_ids or None),
            )
        elif source == "obo":
            merge_rows(rows_by_key, obo_rows(selected_terms, max_ontologies=obo_max_ontologies, skip_ids=bp_ids or None))

    write_rows(out_path, rows_by_key)
    print(f"Done. {len(rows_by_key)} merged rows -> {out_path}")
    return list(rows_by_key.values())


# Run the three-database review. For a quick smoke test, set TERMS = ["disease"], SOURCES = "ols", FETCH_PARENTS = False.
rows = run_review()
rows[:3]

   BioPortal indexes 1286 ontologies — OLS/OBO scan will skip these

-- BioPortal: searching 'disease' --
   240 exact-label hits (264 raw BioPortal hits)
   AD-CDO          Disease
   DDSS            Disease
   AURA            Disease
   CHEAR           Disease
   ESSO            Disease
   BPT             Disease
   PMR             Disease
   CRENO           Disease [BFO: realizable entity (0000017)]
   NEO             disease
   BCGO            Disease
   ABD             Disease
   CDMONTO         Disease
   CANCO           Disease
   COVID-19-ONT-PM Disease
   PMDO            Disease
   ADHER_INTCARE_EN Disease
   NAMO            DISEASE
   HP_O            disease
   OF              Disease
   COVID19-IBO     disease
   DCO             Disease
   DCO             Disease
   CASE-BASE-ONTO  Disease
   IRAEO           Disease
   HTNCO           Disease
   MESH            Disease
   SNOMEDCT        Disease
   LOINC           Disease
   ICPC2P          disease
   EDONTOLOGY      Disease

Done. 253 merged rows -> Scoping review/disease_terms_review.csv


[{'term': 'disease',
  'ontology': 'AD-CDO',
  'preferred_label': 'Disease',
  'class_iri': 'http://www.semanticweb.org/sun/ontologies/AD-CDE#Disease',
  'definition': '',
  'comments': '',
  'synonyms': '',
  'parent_labels': '',
  'parent_iris': '',
  'bfo_anchor': '',
  'mapping_count': 321,
  'source_databases': 'BioPortal',
  'retrieved_at_by_database': 'BioPortal: 2026-06-29T20:06:55-04:00',
  'ontology_versions': 'BioPortal: released=2026-05-21T00:00:00.000+00:00, creationDate=2026-05-21T00:00:00.000+00:00, submissionId=2',
  'source_ontology_ids': 'BioPortal: AD-CDO',
  'source_urls': 'BioPortal: https://data.bioontology.org/ontologies/AD-CDO/classes/http%3A%2F%2Fwww.semanticweb.org%2Fsun%2Fontologies%2FAD-CDE%23Disease',
  'source_records': '[{"source": "BioPortal", "retrieved_at": "2026-06-29T20:06:55-04:00", "ontology": "AD-CDO", "ontology_version": "released=2026-05-21T00:00:00.000+00:00, creationDate=2026-05-21T00:00:00.000+00:00, submissionId=2", "source_url": "https://da